# 🟢 NVIDIA GPU Acceleration — Ketika CPU Tidak Cukup Cepat

**Modul 1: Machine Learning Fundamentals** | Notebook 10 (NVIDIA Spotlight)

---

## 🎯 Apa yang Akan Kita Pelajari?

1. Perbedaan **CPU vs GPU** dan kapan GPU lebih unggul
2. **NVIDIA RAPIDS cuML** — library ML di GPU dengan API identik scikit-learn
3. **XGBoost GPU** — akselerasi XGBoost dengan NVIDIA CUDA
4. Benchmark langsung: seberapa cepat GPU dibanding CPU?

### ⏱️ Estimasi Waktu: ~90 menit + diskusi

---

## 💡 CPU vs GPU — Apa Bedanya?

| | CPU | GPU (NVIDIA) |
|---|---|---|
| **Jumlah core** | 4–16 core | **Ribuan** core (T4 = 2.560 core!) |
| **Kekuatan per core** | Sangat kuat | Lebih sederhana |
| **Cocok untuk** | Tugas sekuensial, logika kompleks | Tugas **paralel** (ribuan operasi sekaligus) |

### Analogi: Restoran

- **CPU** = 1 koki master yang bisa masak apapun, tapi hanya bisa masak 1 piring pada satu waktu
- **GPU** = 1.000 koki sederhana yang masing-masing masak 1 piring bersamaan

Untuk masak **1 piring spesial** → CPU lebih baik.
Untuk masak **1.000 piring sekaligus** → GPU jauh lebih cepat!

Machine Learning = banyak operasi matematika yang **sama** dilakukan pada **ribuan data** → cocok untuk GPU!

### NVIDIA RAPIDS cuML

NVIDIA membuat library **RAPIDS cuML** yang:
- API-nya **identik** dengan scikit-learn (tinggal ganti `from sklearn` → `from cuml`)
- Rata-rata **10–50x lebih cepat** dari CPU
- Mendukung 50+ algoritma ML

Mari kita buktikan sendiri! 🚀

---

## 1️⃣ Setup — Cek GPU dan Install Library

**PENTING:** Notebook ini harus dijalankan dengan **GPU runtime**.

Cara mengaktifkan: `Runtime` → `Change runtime type` → pilih **T4 GPU** → `Save`

In [ ]:
# Cek GPU yang tersedia
!nvidia-smi

In [ ]:
# Install RAPIDS cuML (GPU-accelerated ML library dari NVIDIA)
# Ini mungkin memakan waktu 2-3 menit
!pip install -q cuml-cu12 2>/dev/null

# Verifikasi instalasi
try:
    import cuml
    print(f'✅ RAPIDS cuML versi {cuml.__version__} berhasil diinstall!')
    CUML_AVAILABLE = True
except ImportError:
    print('⚠️ cuML tidak bisa diinstall di environment ini.')
    print('   Demo akan tetap berjalan dengan XGBoost GPU sebagai alternatif.')
    CUML_AVAILABLE = False

In [ ]:
# Import library standar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, r2_score, silhouette_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Helper: konversi output cuML ke numpy (kadang sudah numpy, kadang cuDF)
def to_np(x):
    """Konversi output cuML ke numpy array"""
    if hasattr(x, 'to_numpy'):
        return x.to_numpy()
    return np.asarray(x)

# Dictionary untuk menyimpan hasil benchmark
benchmarks = {}

def benchmark(nama, cpu_time, gpu_time, cpu_score, gpu_score):
    """Simpan dan tampilkan hasil benchmark"""
    speedup = cpu_time / gpu_time if gpu_time > 0 else 0
    benchmarks[nama] = {
        'CPU (detik)': cpu_time,
        'GPU (detik)': gpu_time,
        'Speedup': speedup,
        'CPU Score': cpu_score,
        'GPU Score': gpu_score
    }
    print(f'\n📊 {nama}')
    print(f'  CPU: {cpu_time:.3f} detik (skor: {cpu_score:.4f})')
    print(f'  GPU: {gpu_time:.3f} detik (skor: {gpu_score:.4f})')
    print(f'  ⚡ GPU {speedup:.1f}x lebih cepat!')

print('✅ Library dasar berhasil di-import!')

---

## 2️⃣ Memuat Dataset

Kita menggunakan 2 dataset bawaan scikit-learn (tidak perlu upload file!):

| Dataset | Baris | Fitur | Tugas | Konteks |
|---------|-------|-------|-------|---------|
| **Forest Covertype** | 581.012 | 54 | Klasifikasi (7 jenis pohon) | Prediksi jenis pohon di hutan dari data geografis |
| **California Housing** | 20.640 | 8 | Regresi (harga rumah) | Prediksi harga rumah di California dari data sensus |

Dataset Covertype sengaja dipilih karena **besar** (581K baris) — di sinilah GPU menunjukkan keunggulannya.

In [ ]:
from sklearn.datasets import fetch_covtype, fetch_california_housing

# Dataset 1: Forest Covertype (klasifikasi) — 581K baris
print('⏳ Mengunduh Forest Covertype...')
covtype = fetch_covtype()
X_cov, y_cov = covtype.data, covtype.target

# Label Covertype dimulai dari 1-7, ubah ke 0-6 agar kompatibel dengan XGBoost
y_cov = y_cov - 1

# Split
X_cov_train, X_cov_test, y_cov_train, y_cov_test = train_test_split(
    X_cov, y_cov, test_size=0.2, random_state=42
)

print(f'✅ Forest Covertype: {X_cov.shape[0]:,} baris, {X_cov.shape[1]} fitur, {len(set(y_cov))} kelas')
print(f'   Train: {X_cov_train.shape[0]:,} | Test: {X_cov_test.shape[0]:,}')

# Dataset 2: California Housing (regresi) — 20K baris
print('\n⏳ Mengunduh California Housing...')
housing = fetch_california_housing()
X_house, y_house = housing.data, housing.target

X_house_train, X_house_test, y_house_train, y_house_test = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

print(f'✅ California Housing: {X_house.shape[0]:,} baris, {X_house.shape[1]} fitur')
print(f'   Train: {X_house_train.shape[0]:,} | Test: {X_house_test.shape[0]:,}')

---

## 3️⃣ Benchmark 1: Linear Regression (Regresi)

Algoritma paling dasar — prediksi harga rumah di California.

| scikit-learn (CPU) | cuML (GPU NVIDIA) |
|---|---|
| `from sklearn.linear_model import LinearRegression` | `from cuml.linear_model import LinearRegression` |
| **Kode lainnya identik!** | **Kode lainnya identik!** |

In [ ]:
if CUML_AVAILABLE:
    from sklearn.linear_model import LinearRegression as LR_CPU
    from cuml.linear_model import LinearRegression as LR_GPU

    # --- CPU ---
    start = time.time()
    lr_cpu = LR_CPU()
    lr_cpu.fit(X_house_train, y_house_train)
    score_cpu = r2_score(y_house_test, lr_cpu.predict(X_house_test))
    time_cpu = time.time() - start

    # --- GPU ---
    start = time.time()
    lr_gpu = LR_GPU()
    lr_gpu.fit(X_house_train, y_house_train)
    score_gpu = r2_score(y_house_test, to_np(lr_gpu.predict(X_house_test)))
    time_gpu = time.time() - start

    benchmark('Linear Regression (California Housing)', time_cpu, time_gpu, score_cpu, score_gpu)
    print(f'\n💡 Dataset ini kecil (20K), jadi perbedaan GPU mungkin belum terasa.')
    print(f'   Tunggu benchmark berikutnya dengan data 581K baris!')
else:
    print('⏭️ cuML tidak tersedia — lompat ke benchmark XGBoost GPU')

---

## 4️⃣ Benchmark 2: Random Forest (Klasifikasi)

Random Forest dengan **500 pohon** pada dataset Covertype (581K baris) — di sinilah GPU mulai bersinar!

In [ ]:
if CUML_AVAILABLE:
    from sklearn.ensemble import RandomForestClassifier as RF_CPU
    from cuml.ensemble import RandomForestClassifier as RF_GPU

    n_trees = 200
    print(f'Melatih Random Forest dengan {n_trees} pohon pada {X_cov_train.shape[0]:,} baris...')

    # --- CPU ---
    print('\n⏳ CPU sedang bekerja...')
    start = time.time()
    rf_cpu = RF_CPU(n_estimators=n_trees, max_depth=12, random_state=42, n_jobs=-1)
    rf_cpu.fit(X_cov_train, y_cov_train)
    score_cpu = accuracy_score(y_cov_test, rf_cpu.predict(X_cov_test))
    time_cpu = time.time() - start
    print(f'  CPU selesai: {time_cpu:.1f} detik')

    # --- GPU ---
    print('\n⚡ GPU sedang bekerja...')
    start = time.time()
    rf_gpu = RF_GPU(n_estimators=n_trees, max_depth=12, random_state=42)
    rf_gpu.fit(X_cov_train, y_cov_train)
    score_gpu = accuracy_score(y_cov_test, to_np(rf_gpu.predict(X_cov_test)))
    time_gpu = time.time() - start
    print(f'  GPU selesai: {time_gpu:.1f} detik')

    benchmark('Random Forest 200 Pohon (Covertype 581K)', time_cpu, time_gpu, score_cpu, score_gpu)
else:
    print('⏭️ cuML tidak tersedia — lompat ke benchmark berikutnya')

---

## 5️⃣ Benchmark 3: KNN (Klasifikasi)

K-Nearest Neighbors harus menghitung **jarak ke semua data** — operasi yang sangat cocok untuk GPU karena bisa dihitung paralel.

In [ ]:
if CUML_AVAILABLE:
    from sklearn.neighbors import KNeighborsClassifier as KNN_CPU
    from cuml.neighbors import KNeighborsClassifier as KNN_GPU

    # Gunakan subset agar CPU tidak terlalu lama
    n_samples = 100_000
    X_knn_tr = X_cov_train[:n_samples]
    y_knn_tr = y_cov_train[:n_samples]
    X_knn_te = X_cov_test[:20_000]
    y_knn_te = y_cov_test[:20_000]

    print(f'KNN (K=5) pada {n_samples:,} baris latihan, {len(X_knn_te):,} baris ujian...')

    # --- CPU ---
    print('\n⏳ CPU sedang menghitung jarak ke semua tetangga...')
    start = time.time()
    knn_cpu = KNN_CPU(n_neighbors=5, n_jobs=-1)
    knn_cpu.fit(X_knn_tr, y_knn_tr)
    score_cpu = accuracy_score(y_knn_te, knn_cpu.predict(X_knn_te))
    time_cpu = time.time() - start
    print(f'  CPU selesai: {time_cpu:.1f} detik')

    # --- GPU ---
    print('\n⚡ GPU menghitung ribuan jarak secara paralel...')
    start = time.time()
    knn_gpu = KNN_GPU(n_neighbors=5)
    knn_gpu.fit(X_knn_tr, y_knn_tr)
    score_gpu = accuracy_score(y_knn_te, to_np(knn_gpu.predict(X_knn_te)))
    time_gpu = time.time() - start
    print(f'  GPU selesai: {time_gpu:.1f} detik')

    benchmark('KNN K=5 (Covertype 100K)', time_cpu, time_gpu, score_cpu, score_gpu)
else:
    print('⏭️ cuML tidak tersedia — lompat ke benchmark berikutnya')

---

## 6️⃣ Benchmark 4: K-Means Clustering

Clustering 581K data points ke dalam kelompok-kelompok — GPU menghitung jarak ke centroid untuk **semua data sekaligus**.

In [ ]:
if CUML_AVAILABLE:
    from sklearn.cluster import KMeans as KM_CPU
    from cuml.cluster import KMeans as KM_GPU

    # Scale data untuk clustering
    scaler = StandardScaler()
    X_cov_scaled = scaler.fit_transform(X_cov)

    print(f'K-Means (K=7) pada {X_cov_scaled.shape[0]:,} baris...')

    # --- CPU ---
    print('\n⏳ CPU sedang clustering...')
    start = time.time()
    km_cpu = KM_CPU(n_clusters=7, random_state=42, n_init=10)
    labels_cpu = km_cpu.fit_predict(X_cov_scaled)
    score_cpu = silhouette_score(X_cov_scaled[:10000], labels_cpu[:10000])
    time_cpu = time.time() - start
    print(f'  CPU selesai: {time_cpu:.1f} detik')

    # --- GPU ---
    print('\n⚡ GPU sedang clustering...')
    start = time.time()
    km_gpu = KM_GPU(n_clusters=7, random_state=42, n_init=10)
    labels_gpu = km_gpu.fit_predict(X_cov_scaled)
    score_gpu = silhouette_score(X_cov_scaled[:10000], to_np(labels_gpu)[:10000])
    time_gpu = time.time() - start
    print(f'  GPU selesai: {time_gpu:.1f} detik')

    benchmark('K-Means K=7 (Covertype 581K)', time_cpu, time_gpu, score_cpu, score_gpu)
else:
    print('⏭️ cuML tidak tersedia — lompat ke benchmark berikutnya')

---

## 7️⃣ Benchmark 5: PCA (Dimensionality Reduction)

PCA mengurangi 54 fitur menjadi 10 komponen utama — operasi matriks besar yang sempurna untuk GPU.

In [ ]:
if CUML_AVAILABLE:
    from sklearn.decomposition import PCA as PCA_CPU
    from cuml.decomposition import PCA as PCA_GPU

    if 'X_cov_scaled' not in dir():
        scaler = StandardScaler()
        X_cov_scaled = scaler.fit_transform(X_cov)

    print(f'PCA (54 → 10 komponen) pada {X_cov_scaled.shape[0]:,} baris...')

    # --- CPU ---
    start = time.time()
    pca_cpu = PCA_CPU(n_components=10)
    X_pca_cpu = pca_cpu.fit_transform(X_cov_scaled)
    var_cpu = pca_cpu.explained_variance_ratio_.sum()
    time_cpu = time.time() - start

    # --- GPU ---
    start = time.time()
    pca_gpu = PCA_GPU(n_components=10)
    X_pca_gpu = pca_gpu.fit_transform(X_cov_scaled)
    var_gpu = float(to_np(pca_gpu.explained_variance_ratio_).sum())
    time_gpu = time.time() - start

    benchmark('PCA 54→10 (Covertype 581K)', time_cpu, time_gpu, var_cpu, var_gpu)
    print(f'  (Skor = variance explained ratio)')
else:
    print('⏭️ cuML tidak tersedia — lompat ke benchmark berikutnya')

---

## 8️⃣ Benchmark 6: XGBoost CPU vs GPU

Ini benchmark yang **paling penting** — NVIDIA secara langsung berkontribusi pada pengembangan XGBoost GPU. Cukup ganti `device="cpu"` → `device="cuda"` dan semuanya otomatis berjalan di GPU!

**XGBoost GPU tidak butuh cuML** — ini built-in di XGBoost, jadi benchmark ini **pasti bisa berjalan** di Colab.

In [ ]:
from xgboost import XGBClassifier

print(f'XGBoost (200 pohon) pada {X_cov_train.shape[0]:,} baris...')
print('(Ini mungkin butuh beberapa menit untuk CPU)\n')

# --- CPU ---
print('⏳ XGBoost CPU sedang bekerja...')
start = time.time()
xgb_cpu = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    tree_method='hist', device='cpu',
    random_state=42, verbosity=0
)
xgb_cpu.fit(X_cov_train, y_cov_train)
score_cpu = accuracy_score(y_cov_test, xgb_cpu.predict(X_cov_test))
time_cpu = time.time() - start
print(f'  CPU selesai: {time_cpu:.1f} detik')

# --- GPU ---
print('\n⚡ XGBoost GPU sedang bekerja...')
start = time.time()
xgb_gpu = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    tree_method='hist', device='cuda',   # ← Satu-satunya perubahan!
    random_state=42, verbosity=0
)
xgb_gpu.fit(X_cov_train, y_cov_train)
score_gpu = accuracy_score(y_cov_test, xgb_gpu.predict(X_cov_test))
time_gpu = time.time() - start
print(f'  GPU selesai: {time_gpu:.1f} detik')

benchmark('XGBoost 200 Pohon (Covertype 581K)', time_cpu, time_gpu, score_cpu, score_gpu)
print(f'\n🔑 Perubahan kode: HANYA ganti device="cpu" → device="cuda"')
print(f'   Sisanya 100% identik!')

---

## 9️⃣ Ringkasan Benchmark — Seberapa Cepat GPU?

In [ ]:
# Buat tabel dan grafik ringkasan
if benchmarks:
    df_bench = pd.DataFrame(benchmarks).T
    df_bench.index.name = 'Algoritma'

    print('📊 RINGKASAN BENCHMARK: CPU vs GPU NVIDIA')
    print('=' * 75)
    print(f'{"Algoritma":<40} {"CPU (s)":>10} {"GPU (s)":>10} {"Speedup":>10}')
    print('-' * 75)
    for algo, row in df_bench.iterrows():
        print(f'  {algo:<38} {row["CPU (detik)"]:>10.3f} {row["GPU (detik)"]:>10.3f} {row["Speedup"]:>9.1f}x')
    print('=' * 75)

    # Bar chart speedup
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: Waktu CPU vs GPU
    x = range(len(df_bench))
    width = 0.35
    axes[0].bar([i - width/2 for i in x], df_bench['CPU (detik)'], width,
                label='CPU', color='#EF5350')
    axes[0].bar([i + width/2 for i in x], df_bench['GPU (detik)'], width,
                label='GPU NVIDIA', color='#66BB6A')
    axes[0].set_ylabel('Waktu (detik)', fontsize=12)
    axes[0].set_title('Waktu Training: CPU vs GPU', fontsize=13, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([a.split('(')[0].strip() for a in df_bench.index],
                            rotation=30, ha='right', fontsize=9)
    axes[0].legend(fontsize=11)

    # Plot 2: Speedup
    colors = ['#66BB6A' if s > 1 else '#EF5350' for s in df_bench['Speedup']]
    bars = axes[1].bar(x, df_bench['Speedup'], color=colors)
    axes[1].axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='1x (sama)')
    axes[1].set_ylabel('Speedup (kali lebih cepat)', fontsize=12)
    axes[1].set_title('Berapa Kali GPU Lebih Cepat?', fontsize=13, fontweight='bold')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([a.split('(')[0].strip() for a in df_bench.index],
                            rotation=30, ha='right', fontsize=9)

    for bar, s in zip(bars, df_bench['Speedup']):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f'{s:.1f}x', ha='center', fontsize=11, fontweight='bold')

    axes[1].legend(fontsize=10)
    plt.suptitle('🟢 NVIDIA GPU Acceleration — Benchmark Results',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ Tidak ada benchmark yang berhasil dijalankan.')

In [ ]:
# Verifikasi: apakah akurasi tetap sama?
if benchmarks:
    print('🔍 Verifikasi: Apakah Akurasi Terjaga?')
    print('=' * 65)
    print(f'{"Algoritma":<40} {"CPU Score":>12} {"GPU Score":>12}')
    print('-' * 65)
    for algo, row in df_bench.iterrows():
        match = '✅' if abs(row['CPU Score'] - row['GPU Score']) < 0.01 else '⚠️'
        print(f'  {algo:<38} {row["CPU Score"]:>12.4f} {row["GPU Score"]:>12.4f} {match}')
    print('=' * 65)
    print('\n💡 GPU memberikan KECEPATAN tanpa mengorbankan AKURASI!')
    print('   Perbedaan kecil pada skor adalah normal (beda urutan komputasi floating point).')

---

## 🚀 BONUS: Dari Training ke Deployment — ONNX + NVIDIA GPU Inference

Sejauh ini kita fokus pada **training** (melatih model). Tapi di dunia nyata, model yang sudah dilatih harus **di-deploy** (dipasang) agar bisa dipakai oleh aplikasi.

### Alur Deployment:

```
Train Model (sklearn/XGBoost)
    → Export ke format ONNX (standar terbuka)
    → Jalankan inferensi dengan ONNX Runtime + GPU NVIDIA
    → Siap dipakai di server produksi!
```

### Apa itu ONNX?

**ONNX** (Open Neural Network Exchange) = format standar untuk menyimpan model ML. Keuntungannya:
- Model bisa dilatih di **framework apapun** (sklearn, XGBoost, PyTorch, TensorFlow)
- Lalu dijalankan di **runtime apapun** (CPU, GPU NVIDIA, edge device)
- **ONNX Runtime** dari Microsoft + NVIDIA TensorRT = inferensi super cepat

**Analogi:** ONNX itu seperti format **PDF** — kamu bisa buat dokumen di Word, Google Docs, atau LibreOffice, tapi setelah di-export ke PDF, semua orang bisa membacanya di mana saja.

In [ ]:
# Install library ONNX
!pip install -q onnx skl2onnx onnxruntime-gpu 2>/dev/null || !pip install -q onnx skl2onnx onnxruntime 2>/dev/null

import onnx
import onnxruntime as ort
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

print(f'✅ ONNX Runtime versi {ort.__version__}')
print(f'   Providers tersedia: {ort.get_available_providers()}')

# Gunakan model Random Forest yang sudah dilatih sebelumnya (rf_cpu dari benchmark 2)
# Jika tidak ada, latih baru
try:
    rf_cpu
    print('✅ Menggunakan model Random Forest dari benchmark sebelumnya')
except NameError:
    from sklearn.ensemble import RandomForestClassifier
    print('⏳ Melatih Random Forest untuk demo ONNX...')
    rf_cpu = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
    rf_cpu.fit(X_cov_train, y_cov_train)
    print('✅ Model siap')

# --- Step 1: Export model sklearn ke ONNX ---
print('\n📦 Step 1: Export model sklearn ke format ONNX...')
initial_type = [('input', FloatTensorType([None, X_cov_test.shape[1]]))]
onnx_model = convert_sklearn(rf_cpu, initial_types=initial_type)
onnx.save_model(onnx_model, 'rf_model.onnx')
print(f'  ✅ Model disimpan sebagai rf_model.onnx ({onnx_model.ByteSize()/1024/1024:.1f} MB)')

# --- Step 2: Benchmark inferensi ---
print('\n⚡ Step 2: Benchmark inferensi pada data test ({:,} baris)...\n'.format(len(X_cov_test)))

# Siapkan data test
X_test_float32 = X_cov_test.astype(np.float32)

# 2a. Native sklearn predict (CPU)
start = time.time()
for _ in range(3):  # 3 kali untuk rata-rata
    pred_sklearn = rf_cpu.predict(X_test_float32)
time_sklearn = (time.time() - start) / 3

# 2b. ONNX Runtime CPU
sess_cpu = ort.InferenceSession('rf_model.onnx', providers=['CPUExecutionProvider'])
input_name = sess_cpu.get_inputs()[0].name

start = time.time()
for _ in range(3):
    pred_onnx_cpu = sess_cpu.run(None, {input_name: X_test_float32})[0]
time_onnx_cpu = (time.time() - start) / 3

# 2c. ONNX Runtime GPU (jika tersedia)
gpu_providers = [p for p in ort.get_available_providers()
                 if 'CUDA' in p or 'TensorRT' in p]

if gpu_providers:
    sess_gpu = ort.InferenceSession('rf_model.onnx', providers=gpu_providers + ['CPUExecutionProvider'])
    # Warmup
    sess_gpu.run(None, {input_name: X_test_float32[:100]})

    start = time.time()
    for _ in range(3):
        pred_onnx_gpu = sess_gpu.run(None, {input_name: X_test_float32})[0]
    time_onnx_gpu = (time.time() - start) / 3
    gpu_provider_name = gpu_providers[0]
else:
    time_onnx_gpu = None
    gpu_provider_name = 'Tidak tersedia'

# --- Tampilkan hasil ---
print('📊 Benchmark INFERENSI (prediksi {:,} baris)'.format(len(X_cov_test)))
print('=' * 55)
print(f'  {"Metode":<30} {"Waktu (ms)":>12} {"Speedup":>10}')
print('-' * 55)
print(f'  {"sklearn predict (CPU)":<30} {time_sklearn*1000:>12.1f} {"1.0x":>10}')
print(f'  {"ONNX Runtime (CPU)":<30} {time_onnx_cpu*1000:>12.1f} {time_sklearn/time_onnx_cpu:>9.1f}x')
if time_onnx_gpu is not None:
    print(f'  {"ONNX Runtime (GPU)":<30} {time_onnx_gpu*1000:>12.1f} {time_sklearn/time_onnx_gpu:>9.1f}x')
else:
    print(f'  {"ONNX Runtime (GPU)":<30} {"N/A":>12} {"N/A":>10}')
print('=' * 55)

# Verifikasi akurasi sama
acc_sklearn = accuracy_score(y_cov_test, pred_sklearn)
acc_onnx = accuracy_score(y_cov_test, pred_onnx_cpu)
print(f'\n🔍 Verifikasi akurasi: sklearn={acc_sklearn:.4f}, ONNX={acc_onnx:.4f} ✅')
print(f'   GPU Provider: {gpu_provider_name}')

---

## 📝 Ringkasan: Apa yang Kita Pelajari Hari Ini?

### Fakta Utama

1. **GPU NVIDIA** bisa mempercepat algoritma ML **tanpa mengubah banyak kode**
2. **RAPIDS cuML** menyediakan API yang **identik** dengan scikit-learn — tinggal ganti import
3. **XGBoost GPU** hanya butuh `device="cuda"` — satu baris perubahan!
4. Speedup makin terasa pada **dataset besar** (ratusan ribu baris ke atas)
5. **Akurasi tetap terjaga** — GPU lebih cepat, bukan lebih pintar

### Kapan GPU Berguna?

| Situasi | GPU Berguna? |
|---------|-------------|
| Dataset kecil (<10K baris) | ❌ Overhead GPU lebih besar dari keuntungan |
| Dataset menengah (10K–100K) | ⚠️ Tergantung algoritma |
| Dataset besar (>100K baris) | ✅ GPU mulai menunjukkan keunggulan |
| Dataset sangat besar (>1M baris) | ✅✅ GPU **jauh** lebih cepat |
| Deep Learning | ✅✅✅ GPU **wajib** (akan dipelajari di Modul 2) |

### Ekosistem NVIDIA untuk ML/AI

| Produk | Fungsi |
|--------|--------|
| **RAPIDS cuML** | ML klasik di GPU (pengganti scikit-learn) |
| **cuDF** | Pandas di GPU (pengolahan data cepat) |
| **XGBoost GPU** | Gradient Boosting di GPU |
| **TensorRT** | Optimasi model Deep Learning untuk inferensi |
| **NVIDIA NeMo** | Framework untuk NLP dan AI percakapan |
| **Triton Inference Server** | Deploy model AI di produksi |

💡 Kita akan mempelajari lebih dalam tentang ekosistem NVIDIA di **Modul 6**!

---

### 🔜 Modul Selanjutnya: Deep Learning Fundamentals
Di modul berikutnya, kita akan belajar **Neural Networks** — dan di situlah GPU NVIDIA benar-benar menjadi **kebutuhan mutlak**, bukan sekedar percepatan!